# Pregunta 3 — Detección de Cumplimiento de Normas de Seguridad en Obras de Construcción(YOLO)

Este notebook ejecuta el proyecto **construction_ppe_yolo** en Google Colab.

El objetivo es reproducir de extremo a extremo el pipeline de la Pregunta 3 del examen:

- Descargar datasets desde **Roboflow** usando `ROBOFLOW_API_KEY`;
- Descargar **SHWD** desde Google Drive en formato Pascal VOC/XML;
- Preparar un dataset unificado en formato YOLO;
- Entrenar YOLOv8n baseline, YOLOv8s principal y una ablación sin mosaic;
- Evaluar mAP@0.5, mAP@0.5:0.95 y métricas por clase;
- Ejecutar análisis de oclusión;
- Generar una demostración del verificador de cumplimiento;
- Guardar resultados en `outputs/` y `results/`;
- Respaldar resultados en Google Drive.

> Ejecuta las celdas en orden. Antes de empezar, activa GPU en Colab:  
> **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**.

## 0. Verificar GPU y entorno de Colab

**Qué hace esta celda:**

- Verifica si Colab tiene una GPU activa;
- Muestra el nombre de la GPU y la memoria disponible;
- Confirma la versión de Python.

**Objetivo:** asegurar que el entrenamiento YOLO se ejecute con GPU. Si no aparece GPU, el entrenamiento será mucho más lento.

**Resultado esperado:** un mensaje similar a `GPU: NVIDIA L4` o `GPU: Tesla T4`.

In [ ]:
import sys
import torch

print("Python:", sys.version)

if torch.cuda.is_available():
    print("✓ GPU detectada:", torch.cuda.get_device_name(0))
    print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ No se detectó GPU. Ve a Entorno de ejecución → Cambiar tipo → GPU.")

## 1. Configurar Private API Key de Roboflow

### ¿Qué hace esta celda?

Esta celda solicita la Private API Key de Roboflow y la guarda temporalmente en la variable de entorno:

`ROBOFLOW_API_KEY`

La Private API Key es necesaria para que el proyecto pueda descargar automáticamente los datasets desde Roboflow:

1. Safety Helmet Dataset: `safety-helmet-kfkub/safety-helmet-ifjeb`
2. PPE Detection Dataset: `testcasque/ppe-detection-qlq3d`

### ¿Dónde se ingresa la Private API Key?

Al ejecutar la celda de código que está debajo, Colab mostrará un campo de entrada con el texto:

`ROBOFLOW_API_KEY:`

Debes pegar ahí tu Private API Key de Roboflow y presionar **Enter**.

Por seguridad, el valor que pegues no se mostrará en pantalla.

### Resultado esperado

Si la API Key fue ingresada correctamente, se mostrará:

`✓ ROBOFLOW_API_KEY configurada`

La API Key no se guarda en GitHub ni dentro del notebook. Solo queda activa durante esta sesión de Colab.

In [ ]:
import os
import getpass

if not os.environ.get("ROBOFLOW_API_KEY"):
    os.environ["ROBOFLOW_API_KEY"] = getpass.getpass("ROBOFLOW_API_KEY: ")

if os.environ.get("ROBOFLOW_API_KEY"):
    print("✓ ROBOFLOW_API_KEY configurada")
else:
    raise RuntimeError("No se configuró ROBOFLOW_API_KEY")

## 2. Definir repositorio y carpeta de trabajo

**Qué hace esta celda:**

- Define la URL del repositorio GitHub;
- Define la carpeta local en Colab donde se clonará el proyecto.

**Objetivo:** centralizar la configuración del notebook. Si usas otro repositorio o fork, cambia `REPO_URL`.

**Resultado esperado:** se imprimen `REPO_URL` y `PROJECT_DIR`.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/manadevelop/construction_ppe_yolo.git"
PROJECT_DIR = Path("/content/construction_ppe_yolo")

print("REPO_URL    :", REPO_URL)
print("PROJECT_DIR :", PROJECT_DIR)

## 3. Clonar o actualizar el repositorio

**Qué hace esta celda:**

- Si el repositorio no existe en Colab, lo clona desde GitHub;
- Si ya existe, ejecuta `git pull` para traer los últimos cambios;
- Entra a la carpeta del proyecto.

**Objetivo:** asegurar que Colab use la versión más reciente del código.

**Resultado esperado:** carpeta `/content/construction_ppe_yolo` lista y actualizada.

In [ ]:
import os
import shutil
from pathlib import Path

if PROJECT_DIR.exists():
    print("Repositorio ya existe. Actualizando con git pull...")
    %cd /content/construction_ppe_yolo
    !git pull
else:
    print("Clonando repositorio...")
    !git clone {REPO_URL} {PROJECT_DIR}
    %cd /content/construction_ppe_yolo

print("\nDirectorio actual:", Path.cwd())

## 4. Validar estructura mínima del proyecto

**Qué hace esta celda:**

- Verifica que existan los archivos principales del proyecto;
- Revisa que `run_all.sh`, `requirements.txt`, `configs/`, `scripts/` y `src/` estén presentes.

**Objetivo:** detectar rápidamente si el repositorio está incompleto antes de entrenar.

**Resultado esperado:** todos los archivos principales deben salir con `OK`.

In [ ]:
from pathlib import Path

required_paths = [
    "README.md",
    "requirements.txt",
    "run_all.sh",
    "configs",
    "configs/data_sources.yaml",
    "configs/class_mapping.yaml",
    "scripts",
    "scripts/setup_datasets.py",
    "scripts/prepare_dataset.py",
    "scripts/validate_dataset.py",
    "scripts/train.py",
    "scripts/evaluate.py",
    "scripts/occlusion_analysis.py",
    "scripts/compliance_demo.py",
    "src",
]

missing = []
for p in required_paths:
    path = Path(p)
    status = "OK" if path.exists() else "FALTA"
    print(f"{p:<40} {status}")
    if not path.exists():
        missing.append(p)

if missing:
    raise FileNotFoundError("Faltan archivos/carpetas: " + ", ".join(missing))
else:
    print("\n✓ Estructura mínima validada")

## 5. Revisar fuentes de datos configuradas

**Qué hace esta celda:**

- Imprime el archivo `configs/data_sources.yaml`;
- Permite confirmar qué datasets descargará el pipeline.

**Objetivo:** verificar que las fuentes estén alineadas con la Pregunta 3:

- Safety Helmet Dataset desde Roboflow;
- PPE Detection Dataset desde Roboflow;
- SHWD desde Google Drive, en formato Pascal VOC/XML.

**Resultado esperado:** deben aparecer las fuentes `safety_helmet`, `ppe_detection` y `shwd`.


In [ ]:
from pathlib import Path

path = Path("configs/data_sources.yaml")
print(path.read_text() if path.exists() else "No existe configs/data_sources.yaml")

## 6. Instalar dependencias

**Qué hace esta celda:**

- Instala las dependencias declaradas en `requirements.txt`;
- Valida que estén disponibles los paquetes necesarios para YOLO, Roboflow y descarga desde Google Drive.

**Objetivo:** preparar el entorno antes de descargar datos y entrenar.

**Resultado esperado:** instalación sin errores y validación de `roboflow` y `gdown`.


In [ ]:
%cd /content/construction_ppe_yolo

!pip install -r requirements.txt -q

import roboflow
import gdown
import cv2
import yaml

print("✓ roboflow instalado")
print("✓ gdown instalado")
print("✓ Dependencias principales validadas")


## 7. Ejecutar pipeline completo con un único comando

**Qué hace esta celda:**

Ejecuta:

```bash
bash run_all.sh
```

Ese comando realiza todo el flujo del proyecto:

1. Verificar/instalar dependencias;
2. Descargar Safety Helmet y PPE Detection desde Roboflow;
3. Descargar SHWD desde Google Drive;
4. Convertir SHWD de Pascal VOC/XML a YOLO;
5. Unificar clases en `person`, `helmet`, `no_helmet`, `vest`, `no_vest`;
6. Validar el dataset final YOLO;
7. Entrenar YOLOv8n baseline;
8. Entrenar YOLOv8s principal;
9. Ejecutar ablación `YOLOv8n mosaic=0`;
10. Evaluar mAP@0.5 y mAP@0.5:0.95;
11. Analizar oclusión/dificultad;
12. Generar demo del verificador de cumplimiento;
13. Guardar resumen final en `results/summary.txt`.

**Objetivo:** cumplir el requisito del examen: ejecución reproducible de extremo a extremo con un único comando documentado en el README.

**Resultado esperado:** se crean las carpetas `data/`, `outputs/` y `results/`.

> Esta celda puede tardar varias horas. Si ya existen checkpoints, `run_all.sh` los reutiliza salvo que ejecutes con `FORCE_TRAIN=1`.


In [ ]:
%cd /content/construction_ppe_yolo

# Asegura que el script sea ejecutable
!chmod +x run_all.sh

# Ejecuta el pipeline completo
!bash run_all.sh

## 8. Verificar artefactos generados

**Qué hace esta celda:**

- Revisa que existan los directorios y archivos principales generados por el pipeline;
- Muestra los archivos encontrados en `outputs/` y `results/`.

**Objetivo:** confirmar que el entrenamiento, evaluación, análisis de oclusión y demo de cumplimiento produjeron salidas útiles para el informe.

**Resultado esperado:** deben existir carpetas como:

- `outputs/yolov8n_baseline/`
- `outputs/yolov8s_main/`
- `outputs/ablation_yolov8n_no_mosaic/`
- `results/metrics/`
- `results/occlusion_analysis/`
- `results/compliance_demo/`

In [ ]:
from pathlib import Path

expected = [
    "data/ppe_yolo/data.yaml",
    "outputs/yolov8n_baseline",
    "outputs/yolov8s_main",
    "outputs/ablation_yolov8n_no_mosaic",
    "results",
    "results/metrics",
    "results/occlusion_analysis",
    "results/compliance_demo",
]

for p in expected:
    path = Path(p)
    print(f"{p:<45} {'OK' if path.exists() else 'FALTA'}")

print("\nArchivos principales en results/:")
!find results -maxdepth 3 -type f | sort | head -80

print("\nArchivos principales en outputs/:")
!find outputs -maxdepth 3 -type f | sort | head -80

## 9. Mostrar resumen de métricas

**Qué hace esta celda:**

- Busca archivos `.json`, `.csv` y `.txt` con métricas;
- Imprime su contenido o una vista preliminar.

**Objetivo:** obtener rápidamente los números que irán al informe:

- mAP@0.5;
- mAP@0.5:0.95;
- Métricas por clase;
- Comparación YOLOv8n vs YOLOv8s;
- Efecto de la ablación sin mosaic;
- Resultados de oclusión.

**Resultado esperado:** resumen legible de métricas finales.

In [ ]:
from pathlib import Path
import json
import pandas as pd

metric_files = []

for root in [Path("outputs"), Path("results")]:
    if root.exists():
        for ext in ["*.json", "*.csv", "*.txt", "*.md"]:
            metric_files.extend(root.glob(f"**/{ext}"))

metric_files = sorted(metric_files)

print(f"Archivos de métricas encontrados: {len(metric_files)}\n")

for path in metric_files[:40]:
    print("-" * 90)
    print(path)
    print("-" * 90)

    try:
        if path.suffix == ".json":
            data = json.load(open(path, "r", encoding="utf-8"))
            print(json.dumps(data, indent=2, ensure_ascii=False)[:4000])

        elif path.suffix == ".csv":
            df = pd.read_csv(path)
            display(df.tail(10))

        else:
            print(path.read_text(encoding="utf-8", errors="ignore")[:4000])

    except Exception as e:
        print("No se pudo leer:", e)

## 10. Visualizar figuras generadas

**Qué hace esta celda:**

- Busca imágenes `.png`, `.jpg` y `.jpeg` dentro de `outputs/` y `results/`;
- Muestra las primeras figuras generadas.

**Objetivo:** revisar visualmente:

- Matriz de confusión;
- Curvas de entrenamiento;
- Ejemplos de detección;
- Análisis de oclusión;
- Frames anotados por el verificador de cumplimiento.

**Resultado esperado:** figuras listas para seleccionar e incluir en el informe NeurIPS.


In [ ]:
from pathlib import Path
from IPython.display import Image, display

image_files = []

for root in [Path("outputs"), Path("results")]:
    if root.exists():
        image_files.extend(list(root.glob("**/*.png")))
        image_files.extend(list(root.glob("**/*.jpg")))
        image_files.extend(list(root.glob("**/*.jpeg")))

image_files = sorted(image_files)

print(f"Imágenes encontradas: {len(image_files)}")

for path in image_files[:25]:
    print("\n", path)
    display(Image(filename=str(path)))

## 11. Respaldar `outputs/` y `results/` en Google Drive

**Qué hace esta celda:**

- Monta Google Drive;
- Copia `outputs/` y `results/` a una carpeta persistente en Drive.

**Objetivo:** evitar perder checkpoints, métricas y figuras cuando termine la sesión de Colab.

**Resultado esperado:** resultados copiados en:

```text
/content/drive/MyDrive/EP01/pregunta3/resultados/
```

Puedes cambiar `BACKUP_DIR` si quieres otra ubicación.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import os

drive.mount('/content/drive')

%cd /content/construction_ppe_yolo

BACKUP_DIR = Path('/content/drive/MyDrive/EP01/pregunta3/resultados')
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

for name in ['outputs', 'results']:
    src = Path(name)
    dst = BACKUP_DIR / name

    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"✓ {src} -> {dst}")
    else:
        print(f"⚠ No existe {src}; se omite")

print("\nRespaldo finalizado en:", BACKUP_DIR)